# 05G - End-to-End Preprocessing Pipeline

Build a reusable scikit-learn preprocessing pipeline.

In [1]:

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import joblib

df=pd.read_csv("../data/datasets/american_bankruptcy.csv")
df['target']=df['status_label'].map({'alive':0,'failed':1})
drop=['status_label','target']
if 'company_name' in df.columns:
    drop.append('company_name')
X=df.drop(columns=drop)
y=df['target']

numeric=X.select_dtypes(include='number').columns
categorical=X.select_dtypes(exclude='number').columns

X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.2,random_state=42,stratify=y)

print(X_train.shape,X_test.shape)


(62945, 19) (15737, 19)


## Define Preprocessing Pipelines

In [2]:

numeric_pipeline=Pipeline([
    ('imputer',SimpleImputer(strategy='median')),
    ('scaler',StandardScaler())
])

categorical_pipeline=Pipeline([
    ('imputer',SimpleImputer(strategy='most_frequent'))
])

preprocessor=ColumnTransformer([
    ('num',numeric_pipeline,numeric),
    ('cat',categorical_pipeline,categorical)
])

display(preprocessor)


,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. `

## Fit & Transform

In [3]:

X_train_processed=preprocessor.fit_transform(X_train)
X_test_processed=preprocessor.transform(X_test)

print("Processed train shape:",X_train_processed.shape)
print("Processed test shape:",X_test_processed.shape)


Processed train shape: (62945, 19)
Processed test shape: (15737, 19)


## Persist Pipeline

In [4]:

joblib.dump(preprocessor,"preprocessing_pipeline.joblib")
print("Saved preprocessing_pipeline.joblib")


Saved preprocessing_pipeline.joblib


## Pipeline Summary

In [5]:

summary=pd.DataFrame({
    "Component":["Numeric Imputer","Numeric Scaler","Categorical Imputer"],
    "Method":["Median","StandardScaler","Most Frequent"]
})
display(summary)


,Component,Method
0,Numeric Imputer,Median
1,Numeric Scaler,StandardScaler
2,Categorical Imputer,Most Frequent


## Production Notes

In [6]:

notes=[
"Fit the pipeline only on training data.",
"Reuse the saved pipeline during inference.",
"Bundle this pipeline with the trained model for deployment."
]
for i,n in enumerate(notes,1):
    print(f"{i}. {n}")


1. Fit the pipeline only on training data.
2. Reuse the saved pipeline during inference.
3. Bundle this pipeline with the trained model for deployment.
